#### Functions from Assignment 0

In [27]:
import import_ipynb
from random import randint
from math import sqrt
from sympy import factorint

In [29]:
# Finds the m- ary expansion of n
def getExpansion (n ,m):
    listOfDigits =[]
    while n >= m:
        digit =n%m
        listOfDigits . append ( digit )
        n =(n - digit ) // m
    listOfDigits.append(n)
    return listOfDigits
    
def intToText (n):
    t= getExpansion(n ,256)
    myString =''
    for i in t :
        myString = myString + chr(i)
    return myString

def textToInt (s):
    n =0
    k =0
    for i in s :
        n=n +ord( i) *(256** k)
        k=k +1
    return n

# Output the multiplicative inverse of a modulo p
def multInverse (a , p):
    result = extendedGCD (a , p)
    if result [0]!=1: # Error message if a and p are not relatively prime
        s=" Numbers needs to be relatively prime "
        return s
    inv = result [1]% p
    return inv

#extended euclidean algorithm
# Output [r,s,t] satisfying s*a+t*b=r=gcd(a,b)
def extendedGCD(a , b):
    r0 , r=a ,b
    s0 , s =1 ,0
    t0 , t =0 ,1
    while (r >0):
        tempr , temps , tempt =r ,s ,t
        q= r0 // r
        r ,s , t=r0 - q*r ,s0 -q*s ,t0 - q*t
        r0 , s0 , t0 = tempr , temps , tempt
    return [r0 ,s0 , t0 ]

def fast2Power (a ,n ,m):
    res =1
    while n >0:
        if n %2==1: #If the bit is 1 multiply by the corresponding square
            res =( res *a )%m
        a =( a*a) %m
        n=n //2
    return res

### Functions for Assignment 3

In [31]:
def findSquareRoot (N ,p) :
    N0 = N % p
    if fast2Power (N0, (p - 1) // 2 ,p) == 1: #Euler ’s criterion
        if p % 4 == 3:
            x1 = fast2Power (N0, (p + 1) // 4 , p) #see assignment 2 exercise 2 theoretical part
            y1 = p - x1
            return [x1 , y1]
        else :
            for i in range(1 , (( p - 1) // 2) + 1):
                if (i * i) % p == N0:
                    x1 = i
                    y1 = p - i
                    return [x1 , y1]
    return []

def generateCurve (E , p):
    if isElliptic (E , p) == False :
        print (" This is not an elliptic curve ")
        return None
    A, B = E
    listOfPoints =["O"]
    for x in range (p):
        a =(x**3 + A*x + B) % p
        if a == 0:
            listOfPoints.append ([x ,0])
        if fast2Power (a, (p - 1) // 2, p) == 1: # Euler ’s criterion there are solutions
            y1, y2 = findSquareRoot(a, p)
            listOfPoints.append([x, y1])
            listOfPoints.append([x, y2])
    return listOfPoints

# Part I (common part)

## Implementation part

### Exercise 1

In [64]:
def isElliptic (E ,p ):
    A=E [0]
    B=E [1]
    discr =(4*( A **3) +27*( B **2) )%p
    return discr !=0

def pointOnCurve (P ,E ,p) :
    if P == "O":
        return True
    else :
        A=E [0]
        B=E [1]
        x=P [0]
        y=P [1]
        return (y **2) %p ==( x **3+ A* x+B) %p

In [66]:
E = [0,7] #represents the elliptic curve y^2 = x^3 + 7
p = 2**256 - 2**32 - 977
x, y = "79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798", "483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8"
x10, y10 = int(x, 16), int(y, 16)
P = [x10, y10]
if not isElliptic(E,p):
    print("E is not elliptic!")
elif not pointOnCurve(P, E, p):
    print("P is not on the curve!")
else:
    print("P is on the curve!")

P is on the curve!


### Exercise 2

In [92]:
E = [5,12]
p = 13
C = generateCurve(E, p)
C

['O', [0, 5], [0, 8], [2, 2], [2, 11], [7, 0], [10, 3], [10, 10]]

In [100]:
len(C) <= p + 1 + 2*sqrt(p) and len(C) >= p + 1 - 2*sqrt(p)

True

### Exercise 3

#### (a)

In [79]:
def addPoints(P,Q,E,p):
    A = E[0]
    B = E[1]
    if P == "O":
        return Q
    elif Q == "O":
        return P
    x1, x2 = P[0], Q[0]
    y1, y2 = P[1], Q[1]
    if x1 == x2 % p and y1 == -y2 % p:
        return "O"
    else:
        if P != Q:
            lmbda = (y2 - y1) * multInverse(x2 - x1, p) % p
        else:
            lmbda = (3 * fast2Power(x1, 2, p) + A) * multInverse(2*y1, p) % p
        x3 = (fast2Power(lmbda, 2, p) - x1 - x2) % p
        y3 = (lmbda * (x1 - x3) - y1) % p
        return [x3, y3]

#### (b)

In [101]:
E = [5, 12]
p = 13
P1, Q1 = "O", "O"
P2, Q2 = "O", [0,5]
P3, Q3 = [2, 2], "O"
P4, Q4 = [10, 3], [10, -3]
P5, Q5 = [2, 2], [2, 11]
P6, Q6 = [10, 10], [0, 5]
P7, Q7 = [10, 3], [7, 0]
P8, Q8 = [10, 3], [10, 3]
P9, Q9 = [0, 5], [2, 11]
Plist = [P1, P2, P3, P4, P5, P6, P7, P8, P9]
Qlist = [Q1, Q2, Q3, Q4, Q5, Q6, Q7, Q8, Q9]
for i in range(len(Plist)):
    print(Plist[i], "+", Qlist[i], "=", addPoints(Plist[i], Qlist[i], E, p))

O + O = O
O + [0, 5] = [0, 5]
[2, 2] + O = [2, 2]
[10, 3] + [10, -3] = O
[2, 2] + [2, 11] = O
[10, 10] + [0, 5] = [0, 8]
[10, 3] + [7, 0] = [10, 10]
[10, 3] + [10, 3] = [7, 0]
[0, 5] + [2, 11] = [7, 0]


### Exercise 4